In [ ]:

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import median_absolute_error
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader


In [ ]:


# 定义文件路径
train_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/mohs_hardness/train.csv'
test_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/mohs_hardness/test.csv'

# 使用pandas读取数据
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# 显示前几行以检查数据
train_df.head(), test_df.head()



(     id  allelectrons_Total  ...  density_Average  Hardness
 0  2124                30.0  ...          0.51006       6.0
 1   394                64.0  ...          4.74000       3.3
 2  3101                97.0  ...          1.79976       5.3
 3  1737               151.0  ...          7.77500       1.8
 4   561               131.0  ...          1.92652       5.5
 
 [5 rows x 13 columns],
      id  allelectrons_Total  ...  density_Average  Hardness
 0  3952                80.0  ...          0.86724       5.5
 1  4597               239.0  ...          1.86481       7.0
 2  9621               152.0  ...          0.51227       2.5
 3  8620               232.0  ...          0.59761       6.8
 4  4294                46.0  ...          4.01089       2.5
 
 [5 rows x 13 columns])

In [ ]:


# 定义特征列和目标列
feature_columns = train_df.columns[1:-1]  # 假设第一列是id，最后一列是Hardness
target_column = 'Hardness'

# 提取特征和目标
X_train = train_df[feature_columns]
y_train = train_df[target_column]

X_test = test_df[feature_columns]
y_test = test_df[target_column]

# 数据标准化
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 转换为PyTorch张量
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

# 查看标准化后的数据
X_train_tensor[:5], y_train_tensor[:5]


Output truncated. Showing the last 2800 characters. You should try again and use computer.ai.summarize(output) over the output, or break it down into smaller steps.

\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\ipykernel\kernelbase.py", line 778, in execute_request
    reply_content = await reply_content
  File "D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\ipykernel\ipkernel.py", line 449, in do_execute
    res = shell.run_cell(
  File "D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\ipykernel\zmqshell.py", line 549, in run_cell
    return super().run_cell(*args, **kwargs)
  File "D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\IPython\core\interactiveshell.py", line 3075, in run_cell
    result = self._run_cell(
  File "D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\IPython\core\interactiveshell.py", line 3130, in _run_cell
    

In [ ]:



# 定义神经网络模型
class HardnessPredictor(nn.Module):
    def __init__(self, input_dim):
        super(HardnessPredictor, self).__init__()
        self.fc1 = nn.Linear(input_dim, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)
        self.tanh = nn.Tanh()

    def forward(self, x):
        x = self.tanh(self.fc1(x))
        x = self.tanh(self.fc2(x))
        x = self.fc3(x)
        return x

# 初始化模型
input_dim = X_train_tensor.shape[1]
model = HardnessPredictor(input_dim)

# 定义损失函数和优化器
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

# 查看模型结构
model


HardnessPredictor(
  (fc1): Linear(in_features=11, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=32, bias=True)
  (fc3): Linear(in_features=32, out_features=1, bias=True)
  (tanh): Tanh()
)

In [ ]:



# 定义训练函数
def train_model(model, X_train, y_train, criterion, optimizer, num_epochs=100, batch_size=32):
    dataset = torch.utils.data.TensorDataset(X_train, y_train)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

    for epoch in range(num_epochs):
        running_loss = 0.0
        for inputs, labels in dataloader:
            # 清零梯度
            optimizer.zero_grad()
            # 前向传播
            outputs = model(inputs)
            # 计算损失
            loss = criterion(outputs, labels)
            # 反向传播
            loss.backward()
            # 更新权重
            optimizer.step()
            running_loss += loss.item()

        # 打印每个epoch的平均损失
        if (epoch + 1) % 10 == 0:
            print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {running_loss / len(dataloader):.4f}')

# 训练模型
train_model(model, X_train_tensor, y_train_tensor, criterion, optimizer, num_epochs=100)



Epoch [10/100], Loss: 1.7744
Epoch [20/100], Loss: 1.7253
Epoch [30/100], Loss: 1.6982
Epoch [40/100], Loss: 1.6761
Epoch [50/100], Loss: 1.6606
Epoch [60/100], Loss: 1.6544
Epoch [70/100], Loss: 1.6403
Epoch [80/100], Loss: 1.6215
Epoch [90/100], Loss: 1.6076
Epoch [100/100], Loss: 1.5998


In [ ]:



# 在测试集上进行预测
model.eval()  # 设置模型为评估模式
with torch.no_grad():
    y_pred = model(X_test_tensor).squeeze()

# 计算中位绝对误差
medae = median_absolute_error(y_test_tensor, y_pred)
medae


Output truncated. Showing the last 2800 characters. You should try again and use computer.ai.summarize(output) over the output, or break it down into smaller steps.

n\metrics\_regression.py:900, in median_absolute_error(y_true, y_pred, multioutput, sample_weight)
    835 @validate_params(
    836     {
    837         "y_true": ["array-like"],
   (...)
    845     y_true, y_pred, *, multioutput="uniform_average", sample_weight=None
    846 ):
    847     """Median absolute error regression loss.
    848 
    849     Median absolute error output is non-negative floating point. The best value
   (...)
    898     np.float64(0.85)
    899     """
--> 900     y_type, y_true, y_pred, multioutput = _check_reg_targets(
    901         y_true, y_pred, multioutput
    902     )
    903     if sample_weight is None:
    904         output_errors = np.median(np.abs(y_pred - y_true), axis=0)

File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\sklearn\metrics\_

In [ ]:



# 计算中位绝对误差
with torch.no_grad():
    abs_errors = torch.abs(y_pred - y_test_tensor)
    medae = torch.median(abs_errors).item()

medae



1.4179377555847168